# Solution B1 · How many components? PCA in a pipeline on the digits

**Type**: Implement with scikit-learn

## Objective

Choose the number of components of a PCA in a `StandardScaler` → `PCA` → `LogisticRegression` `Pipeline` with cross-validation, compare with the model without reduction, and find out that **standardizing the pixels changes how many components are needed**: `PCA(0.95)` chooses
a different number depending on the scaler (and from one fold to another).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
ks = [2, 5, 10, 15, 20, 30, 40, 64]

## Commented solution

In [ ]:
def pipeline(k, scale=True):
    steps = ([StandardScaler()] if scale else []) + [PCA(k), LogisticRegression(max_iter=3000)]
    return make_pipeline(*steps)


acc_k = pd.Series({k: cross_val_score(pipeline(k), X, y, cv=cv).mean() for k in ks})
acc_k.round(4)

In [ ]:
acc_without_pca = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
rule_k = min(k for k in ks if acc_k[k] >= acc_k.max() - 0.01)
rule_k_loss = acc_without_pca - acc_k[rule_k]
print(f"without PCA {acc_without_pca:.4f} | k of the rule (within 0.01 of the maximum): {rule_k} components out of 64, accuracy {acc_k[rule_k]:.4f} (loss {rule_k_loss:.4f})")

In [ ]:
r_scaled = cross_validate(pipeline(0.95), X, y, cv=cv, return_estimator=True)
r_raw = cross_validate(pipeline(0.95, scale=False), X, y, cv=cv, return_estimator=True)
scaled_k_per_fold = [e[-2].n_components_ for e in r_scaled["estimator"]]           # the second-to-last step is the PCA
raw_k_per_fold = [e[-2].n_components_ for e in r_raw["estimator"]]
raw_acc = r_raw["test_score"].mean()
k_difference = np.mean(scaled_k_per_fold) - np.mean(raw_k_per_fold)
print("components for 95% of the variance, per fold | with scaler:", scaled_k_per_fold, "| without scaler:", raw_k_per_fold)
print(f"accuracy without scaler: {raw_acc:.4f} | extra components with the scaler: {k_difference:.1f}")

## Check

In [ ]:
with verify():
    check(list(acc_k.index) == ks and acc_k[2] < 0.6 and acc_k[64] > 0.96, "curve over eight values of k: with 2 components below 0.6, with 64 above 0.96", "pipeline with StandardScaler, PCA(k), logistic")
    check(acc_k.is_monotonic_increasing, "the accuracy grows with k", "more components, more information")
    check(acc_without_pca > 0.96 and abs(acc_without_pca - acc_k[64]) < 1e-9, "with 64 components (no reduction) the accuracy is the one without PCA: a rotation does not change the logistic", "PCA(64) is just a rotation of the axes")
    check(rule_k == min(k for k in ks if acc_k[k] >= acc_k.max() - 0.01) and 30 <= rule_k <= 40 and 0 < rule_k_loss < 0.012, "k of the rule: 30-40 components out of 64, with a loss below 1.2 points", "the smallest k within 0.01 of the maximum")
    check(len(scaled_k_per_fold) == 5 and all(35 <= k <= 45 for k in scaled_k_per_fold) and all(24 <= k <= 32 for k in raw_k_per_fold), "PCA(0.95): 35-45 components with the scaler, 24-32 without", "cross_validate(..., return_estimator=True) and pipeline[k].n_components_")
    check(k_difference > 8 and raw_acc > 0.94, "the scaler requires over 8 more components for the same 95% of variance; without the scaler the accuracy stays above 0.94", "means of k_per_fold")
done()

## What you learned

- **`k` is a hyperparameter:** the accuracy grows with `k` and flattens (0.54 with 2 components, 0.94 with 20, 0.963 with 40, 0.969 with 64). With the rule "the smallest `k` within one point of the maximum" you keep 40 components out of 64 while losing less than a point.
- **PCA did not improve the accuracy** (0.969 without reduction): with a logistic there is no noise to remove and a rotation adds no information. The gain is **compression** (fewer dimensions, less memory and downstream time), at almost no cost.
- **`PCA(0.95)` is a rule, not a number:** with the scaler you need to keep about 40 components (39-40 depending on the fold), without the scaler 28 (28-29). Standardizing the pixels gives the same weight to the almost always dark ones (edges), which are largely noise, and the variance
  spreads over more components. Scaling is not "always right" (M5 F1 showed the same for distances): on images with homogeneous pixels it is often not needed.
- PCA inside the `Pipeline` is redone at every fold: the number of components can vary (here by one). With a fixed `k` the behavior is more predictable; with a threshold the final model has a number of components that depends on the training data.

## Common pitfall

Computing PCA on the whole dataset "to decide `k`" and then putting `PCA(k)` in cross-validation. It is almost harmless with PCA (it does not look at `y`) but the chosen number already depends on the validation folds: when in doubt, `GridSearchCV` on the parameter `pca__n_components` inside the pipeline.

## Interview question

<details><summary>How do you choose the number of principal components?</summary>

It depends on the use. For visualization 2-3. For reconstruction a threshold on the cumulative variance or the required reconstruction error. For a downstream model I treat `k` as a hyperparameter and search it in cross-validation inside the pipeline, with a parsimony rule
(the smallest k within the standard error of the best). I remember that the explained variance measures reconstruction fidelity, not usefulness for the target.

</details>